# Domain-Adaptive GPT-2 — Phase 2: RAG Supervised Fine-Tuning

Fine-tunes the phase-1 clinical base on **retrieved evidence + patient summary -> structured report** pairs so the generator learns report structure and evidence reuse (grounded style) instead of memorized medical facts.

Input: `data/processed/sft_pairs.jsonl` with fields
`{"patient": {...}, "prompt": "...", "target": "FINDINGS:..."}`
built locally by `scripts/make_sft_data.py` (commit the file to the repo or push it into Colab).

Output checkpoint: `models/gpt2-clinical-rag` (copied into the repository or downloaded locally for inference).

In [ ]:
!nvidia-smi -L
!pip -q install transformers datasets accelerate sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
REPO = '/content/medical-cdss'
DRIVE = '/content/drive/MyDrive/medical-cdss'
BASE = f'{DRIVE}/gpt2-clinical-base'      # phase-1 checkpoint
OUT = f'{DRIVE}/gpt2-clinical-rag'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.isdir(REPO):
    !git clone https://github.com/<ORG-OR-USER>/medical-cdss.git $REPO
%cd $REPO

In [ ]:
import json
from pathlib import Path

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

SFT_FILE = Path(f'{REPO}/data/processed/sft_pairs.jsonl')
assert SFT_FILE.exists(), (
    'SFT pairs missing - run scripts/make_sft_data.py locally first and upload '
    'data/processed/sft_pairs.jsonl to the repo or Colab.')

pairs = [json.loads(l) for l in open(SFT_FILE)]
print(f'{len(pairs)} training pairs')
print(pairs[0]['target'][:300])

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE)

MAX_LEN = 1024
def to_features(rec):
    prompt, target = rec['prompt'], rec['target']
    p_ids = tok(prompt, truncation=True, max_length=MAX_LEN - 160)['input_ids']
    t_ids = tok(target, truncation=True, max_length=150)['input_ids'] + [tok.eos_token_id]
    input_ids = p_ids + t_ids
    labels = ([-100] * len(p_ids)) + t_ids   # supervise only on the report
    return {'input_ids': input_ids, 'labels': labels}

ds = Dataset.from_list(pairs).map(to_features, remove_columns=Dataset.from_list(pairs).column_names)

def pad_batch(batch):
    import torch
    maxlen = max(len(x) for x in batch['input_ids'])
    def fix(seq, fill):
        return [seq + [fill] * (maxlen - len(seq))]
    return {'input_ids': torch.tensor(sum((fix(s, tok.pad_token_id) for s in batch['input_ids']), [])),
            'attention_mask': torch.tensor(sum((fix([1] * len(s), 0) for s in batch['input_ids']), [])),
            'labels': torch.tensor(sum((fix(l, -100) for l in batch['labels']), []))}

ds = ds.shuffle(seed=19)
split = ds.train_test_split(test_size=0.1)
print(split)

In [ ]:
args = TrainingArguments(
    output_dir=f'{DRIVE}/ckpt-clinical-rag',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=pad_batch,
)
trainer.train()
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('saved RAG generator ->', OUT)

In [ ]:
# in-notebook smoke test of the grounding prompt format
import json
from src.generation.prompting import build_prompt, GenerationConfig
from src.retrieval.hybrid import build_patient_query, hybrid_search

with open(f'{REPO}/configs/sample_patient.json') as fh:
    patient = json.load(fh)
evidence = hybrid_search(build_patient_query(patient))
prompt = build_prompt(patient, evidence, GenerationConfig.from_settings())

import torch
enc = tok(prompt, return_tensors='pt', truncation=True, max_length=1024)
out = model.generate(**enc, max_new_tokens=250, do_sample=True,
                     temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True)[len(prompt):])

Copy `gpt2-clinical-rag` into `models/` of a local clone, then run
`python -m src.generation.cli --patient configs/sample_patient.json`
and the ablation runner in `src/evaluation/ablation.py`.